### Importing necessary modules

In [1]:
import pandas as pd                                                                     # type: ignore
import matplotlib.pyplot as plt                                                         # type: ignore
import seaborn as sns                                                                   # type: ignore
import numpy as np                                                                      # type: ignore
import tensorflow as tf                                                                 # type: ignore
import matplotlib as mpl                                                                # type: ignore
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
mpl.rcParams['figure.figsize'] = (9, 7)
from joblib import dump, load                                                           # type: ignore
import random                                                                           # type: ignore
import cartopy.crs as ccrs                                                              # type: ignore
import cartopy.feature as cfeature                                                      # type: ignore
#np.random.seed(42)                                      
from netCDF4 import Dataset                                                             # type: ignore

2024-08-18 17:18:26.134114: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-18 17:18:26.443249: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-18 17:18:27.462763: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-18 17:18:29.588166: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
geodata = Dataset("/home/mendrika/mendrika-phd/codes/nflics/geoloc_grids/nxny1640_580_nxnyds164580_blobdx0.04491576_area4_n23_20_32.nc")

location_lat = 18.166
location_lon = 8.0

In [3]:
step = 0.1

longitude = np.arange(5, 20+step, step)
latitude = np.arange(12, 25+step, step)

lons, lats = np.meshgrid(longitude, latitude)

In [4]:
import sys
sys.path.insert(1, "/home/mendrika/mendrika-phd/codes/nflics")
import nflics  

### Importing dataset

### Exploratory data analysis

In [5]:
def log_transform(df, keys):
    df_copy = df.copy()    
    for key in keys:
        transformed = []
        for i in df_copy[key]:
            if i > 0:
                transformed.append(np.log(i))
            else:
                transformed.append(np.log(i+1e-8))    
        df_copy[key] = transformed
    return df_copy

In [6]:
def sqrt_transform(df, keys):
    df_copy = df.copy()    
    for key in keys:
        transformed = []
        for i in df_copy[key]:
            if i >= 0:
                transformed.append(np.sqrt(i)) 
        df_copy[key] = transformed
    return df_copy

In [7]:
# Presence or absence of convection in Dakar at time t+1
target_index = "Cb_Dakar"

# Input at time t0
t0 = "year,month,day,hour,minute,"

# latitude and longitude, wavelet power, storm size and distance to Dakar
location = ""
wavelet_power = ""
storm_size = ""
distance = ""
for i in range(1,6):
    location += f"lat{i},lon{i},"
    wavelet_power += f"wp{i},"
    distance += f"ds{i},"
    storm_size += f"size{i},"

# combining all fields to form the feature input
features = t0 + location + wavelet_power + storm_size + distance
field =  features + target_index
field = field.split(',')

In [8]:
to_scale = wavelet_power + storm_size + distance

Computing the average wavelet power

In [9]:
mean_wp = 3430.40
median_wp = 2266.0

mean_size = 7460.30
median_size = 5000.0

In [10]:
def create_storm():
    return {
        'year': [], 
        'month': [], 
        'day': [], 
        'hour': [], 
        'minute': [], 
        'lat1': [], 
        'lon1': [], 
        'lat2': [], 
        'lon2': [], 
        'lat3': [], 
        'lon3': [], 
        'lat4': [], 
        'lon4': [], 
        'lat5': [], 
        'lon5': [], 
        'wp1': [], 
        'wp2': [], 
        'wp3': [], 
        'wp4': [], 
        'wp5': [], 
        'size1': [], 
        'size2': [], 
        'size3': [], 
        'size4': [], 
        'size5': [], 
        'ds1': [], 
        'ds2': [], 
        'ds3': [], 
        'ds4': [], 
        'ds5': [], 
    }

For reproducibility

In [11]:
def X0(input_latitude, input_longitude):
    # Assume these are the minimum values for your grid
    min_latitude = 3.34
    min_longitude = -23.91

    # Step size
    step = 0.03

    # Calculate the index
    latitude_index = int(round((input_latitude - min_latitude) / step))
    longitude_index = int(round((input_longitude - min_longitude) / step))

    return latitude_index, longitude_index

In [12]:
rng1 = np.random.RandomState(26)
rng2 = np.random.RandomState(21)
def generate_geo_coords(lat_min, lat_max, lon_min, lon_max):
    artificial_storm_lat = round(rng1.uniform(lat_min, lat_max), 6)
    artificial_storm_lon = round(rng1.uniform(lon_min, lon_max), 6)
    return artificial_storm_lat, artificial_storm_lon

In [13]:
def distance_to_location(location_lat, location_lon, input_lat, input_lon):    
    location_y, location_x = nflics.X0(location_lat, location_lon)
    if nflics.X0(input_lat, input_lon):
        y, x = nflics.X0(input_lat, input_lon)
    else:
        y, x = X0(input_lat, input_lon)
    return round(np.sqrt((y - location_y)**2 + (x - location_x)**2), 4)

In [14]:
number_of_data_points = 1
number_of_storms_to_generate = 1
number_of_storms_in_model = 5

In [15]:
def generate_storm_cluster_from_a_center(lat_center, lon_center):    
    lat_clusters = []
    lon_clusters = []    
    for _ in range(5):
        lat_candidate = lat_center + random.normalvariate(0,0.2)    
        lat_clusters.append(lat_candidate)    
        lon_candidate = lon_center + random.normalvariate(0,0.2)
        lon_clusters.append(lon_candidate)
    return lat_clusters, lon_clusters

In [16]:
artificial_storms = create_storm()
for lat_center, lon_center in zip(lats.ravel()[:], lons.ravel()[:]):
    for _ in range(number_of_data_points):
        year, month, day, hour, minute = 2020, 7, 15, 18, 0             # need to change this back to 21 (depending on the distribution)
        artificial_storms["year"].append(year)
        artificial_storms["month"].append(month)
        artificial_storms["day"].append(day)
        artificial_storms["hour"].append(hour)
        artificial_storms["minute"].append(minute)
        lat_candidates, lon_candidates = generate_storm_cluster_from_a_center(lat_center, lon_center)
        for j in range(1, number_of_storms_in_model + 1):                        
            artificial_storm_lat, artificial_storm_lon = lat_candidates[j-1], lon_candidates[j-1]  #field name indices start from 1
            artificial_storms[f"lat{j}"].append(artificial_storm_lat)
            artificial_storms[f"lon{j}"].append(artificial_storm_lon)
            d_to_location = distance_to_location(location_lat, location_lon, artificial_storm_lat, artificial_storm_lon)
            artificial_storms[f"ds{j}"].append(d_to_location)
            artificial_storms[f"wp{j}"].append(median_wp)
            artificial_storms[f"size{j}"].append(median_size)        

raw_artificial_storm_data = pd.DataFrame.from_dict(artificial_storms, orient='columns')
artificial_storm_data = log_transform(raw_artificial_storm_data, to_scale.split(',')[:-1])
artificial_storm_data = sqrt_transform(artificial_storm_data, t0.split(',')[:-1])

raw_artificial_storm_data.to_csv("test-data-from-latlon-map-AirMountain-0.1-corrected-1800.csv", index=False)

/home/mendrika/anaconda3/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/mendrika/anaconda3/lib/python3.10/site-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


In [17]:
def find_grid_index(latitudes, longitudes, input_lat, input_lon):
    num_rows, num_cols = latitudes.shape
    
    # Flatten the latitudes and longitudes for easier vector operations
    flat_lat = latitudes.flatten()
    flat_lon = longitudes.flatten()
    
    # Create a matrix of the grid cell boundaries
    lat_min = np.minimum(latitudes[:-1, :-1], latitudes[1:, :-1])
    lat_max = np.maximum(latitudes[:-1, :-1], latitudes[1:, :-1])
    
    lon_min = np.minimum(longitudes[:-1, :-1], longitudes[:-1, 1:])
    lon_max = np.maximum(longitudes[:-1, :-1], longitudes[:-1, 1:])
    
    # Check if the input coordinates fall within any grid cell
    lat_cond = (lat_min <= input_lat) & (input_lat <= lat_max)
    lon_cond = (lon_min <= input_lon) & (input_lon <= lon_max)
    
    # Find the indices where both conditions are true
    valid_indices = np.argwhere(lat_cond & lon_cond)
    
    if valid_indices.size == 0:
        return None, None  # If no cell contains the coordinates
    
    # Return the first valid index (or handle multiple matches if needed)
    row, col = valid_indices[0]
    return row, col


In [18]:
find_grid_index(geodata["lats_mid"][:], geodata["lons_mid"][:], location_lat, location_lon)

(504, 1049)

In [19]:
find_grid_index(geodata["lats_mid"][:], geodata["lons_mid"][:], 13, 12)

(324, 1198)

In [20]:
file = "/home/mendrika/mendrika-phd/data/2020-07-15/Hist_cores_wa_202007150000.nc"
data = Dataset(file, mode='r')

cores = data["msg_cores"][:]
raw_cores = np.copy(cores)


In [21]:
lats.shape

(131, 151)

In [22]:
np.logspace(-5, 2, 4)

array([1.00000000e-05, 2.15443469e-03, 4.64158883e-01, 1.00000000e+02])